# Ratemap Extraction and Caching

Precomputes ratemaps for every model checkpoint and stores them in a memory mapped
.npy cache, so the analysis notebooks (Individual_exp, Group_exp) can load them instead
of recomputing. Run this first.

For each experiment, walks every training epoch across the three environments: loads
the checkpoint, computes embeddings, builds ratemaps, and writes them to the cache at a
running global index. Total checkpoints = `num_epochs * len(envs) + 1` (301), since the
first environment also includes the untrained model (trial 0).

The `control` flag picks the condition: True reads Tmaze_v0a/b/c and writes to
`Models/Experiment1/Control-Exp{n}_cache_ratemaps/`; False reads Tmaze_0/1/2 and the `Experimental-`
folder. Single Experiment runs one experiment; Multiple Experiment loops over a batch.

### Import libraries

In [1]:
import sys
import pathlib

### Import Autoencoder class

In [3]:
folder_path = str(pathlib.Path().resolve().parents[1])
print("Folder path =", folder_path)
sys.path.append(folder_path + '/Notebooks/Spatial_AE/')

from data import *
from training import *
from analysis import *

Folder path = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps


### Check GPU

In [5]:
print("CUDA is available =", torch.cuda.is_available())
print("CUDA device count =", torch.cuda.device_count())
if torch.cuda.device_count() > 1:
    os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
print("CUDA device name =", torch.cuda.get_device_name(0))

CUDA is available = True
CUDA device count = 2
CUDA device name = NVIDIA RTX A5000


### Hyperparameters

In [7]:
#Condition
control = True

#Model
n_hidden = 200
batch_size = 256
num_epochs = 100
learning_rate = 1e-4
C_factor = 1e3
alpha = 1e5

#Others
experiment_n = 1
trial_t = 0

if control == True:
    envs = ['Tmaze_v0a', 'Tmaze_v0b', 'Tmaze_v0c']
else:
    envs = ['Tmaze_0', 'Tmaze_1', 'Tmaze_2']

## Single Experiment Ratemaps Extraction

In [18]:
if control == True:
    CACHE_DIR = folder_path + "/Models/Experiment1/Control-Exp" + str(experiment_n) + "_cache_ratemaps"
else:
    CACHE_DIR = folder_path + "/Models/Experiment1/Experimental-Exp" + str(experiment_n) + "_cache_ratemaps"

os.makedirs(CACHE_DIR, exist_ok=True)

N_BINS = 50
FILTER_WIDTH = 3

# Trials/checkpoints are 0..num_epochs inclusive in your logic (since you use num_epochs+1 sometimes)
n_checkpoints = num_epochs * len(envs) + 1

cache_path = os.path.join(CACHE_DIR, "ratemaps.npy")
cache_arr = open_ratemap_cache(
    cache_path,
    n_checkpoints=n_checkpoints,
    n_units=n_hidden,
    h=N_BINS,
    w=N_BINS,
    dtype=np.float32
)

global_t = 0
foldername = 'Exp' + str(experiment_n) + '-NEpisodes' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate)

for m in range(len(envs)):
    env = envs[m]
    dataset, csv_filename = Load_env_dataset(folder_path, env)
    _, position = update_data(csv_filename)

    env_trial_range = num_epochs +1 if m == 0 else num_epochs

    for t in range(env_trial_range):
        env_t = t+1 if m != 0 else t
        print(f"Tmaze={m}, Trial={env_t}, Global={global_t}")

        model_filename =  env + '-Trial' + str(env_t) + 'of' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate) + '.pth'
        
        model = update_model_trial(control, experiment_n, env, env_t, model_filename, foldername, folder_path, n_hidden)
        embeddings = get_latent_vectors(dataset, model, batch_size=batch_size)

        rmaps = ratemaps(embeddings, position, n_bins=N_BINS, filter_width=FILTER_WIDTH)
        rmaps = np.asarray(rmaps, dtype=np.float32)

        # skip if already written
        if not np.isnan(cache_arr[global_t, 0, 0, 0]):
            continue

        cache_arr[global_t] = rmaps

        global_t = global_t + 1

del cache_arr  # flush once at the end

----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = Tmaze_v0a
Num. Images in Dataset = 39998
Dataset Path = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0a
CSV path     = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0a/data.csv

Dataset Shape  = (39998, 120, 160, 3)
Position Shape = (39998, 2)
Tmaze=0, Trial=0, Global=0
Tmaze=0, Trial=1, Global=1
Tmaze=0, Trial=2, Global=2


KeyboardInterrupt: 

## Multiple Experiment Ratemaps Extraction

In [10]:
batch_training = True

In [11]:
if batch_training == True:
    total_experiments = 20
    starting_experiment = 1
    current_exp = starting_experiment
    
    while current_exp <= total_experiments:
        if control == True:
            CACHE_DIR = folder_path + "/Models/Experiment1/Control-Exp" + str(current_exp) + "_cache_ratemaps"
        else:
            CACHE_DIR = folder_path + "/Models/Experiment1/Experimental-Exp" + str(current_exp) + "_cache_ratemaps"

        os.makedirs(CACHE_DIR, exist_ok=True)
        
        N_BINS = 50
        FILTER_WIDTH = 3
        
        
        # Trials/checkpoints are 0..num_epochs inclusive in your logic (since you use num_epochs+1 sometimes)
        n_checkpoints = num_epochs * len(envs) + 1
        
        cache_path = os.path.join(CACHE_DIR, "ratemaps.npy")
        cache_arr = open_ratemap_cache(
            cache_path,
            n_checkpoints=n_checkpoints,
            n_units=n_hidden,
            h=N_BINS,
            w=N_BINS,
            dtype=np.float32
        )
        
        global_t = 0
        foldername = 'Exp' + str(current_exp) + '-NEpisodes' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate)
        
        for m in range(len(envs)):
            env = envs[m]
            dataset, csv_filename = Load_env_dataset(folder_path, env)
            _, position = update_data(csv_filename)
        
            env_trial_range = num_epochs +1 if m == 0 else num_epochs
        
            for t in range(env_trial_range):
                env_t = t+1 if m != 0 else t
                print(f"Tmaze={m}, Trial={env_t}, Global={global_t}")

                model_filename =  env + '-Trial' + str(env_t) + 'of' + str(num_epochs) + '-NHidden' + str(n_hidden) + '-BSize' + str(batch_size) + '-C' + str(C_factor) + '-A' + str(alpha) + '-LR' + str(learning_rate) + '.pth'
        
                model = update_model_trial(control, current_exp, env, env_t, model_filename, foldername, folder_path, n_hidden)
                embeddings = get_latent_vectors(dataset, model, batch_size=batch_size)
        
                rmaps = ratemaps(embeddings, position, n_bins=N_BINS, filter_width=FILTER_WIDTH)
                rmaps = np.asarray(rmaps, dtype=np.float32)
        
                # skip if already written
                if not np.isnan(cache_arr[global_t, 0, 0, 0]):
                    continue
        
                cache_arr[global_t] = rmaps
        
                global_t = global_t + 1
        
        del cache_arr  # flush once at the end
        
        current_exp += 1

----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = Tmaze_v0a
Num. Images in Dataset = 39998
Dataset Path = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0a
CSV path     = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0a/data.csv

Dataset Shape  = (39998, 120, 160, 3)
Position Shape = (39998, 2)
Tmaze=0, Trial=0, Global=0
Tmaze=0, Trial=1, Global=1
Tmaze=0, Trial=2, Global=2
----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = Tmaze_v0b
Num. Images in Dataset = 39998
Dataset Path = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0b
CSV path     = C:\Users\specs\Desktop\Robotology\Repos\RORcogmaps/Datasets/Tmaze_v0b/data.csv

Dataset Shape  = (39998, 120, 160, 3)
Position Shape = (39998, 2)
Tmaze=1, Trial=1, Global=3
Tmaze=1, Trial=2, Global=4
Tmaze=1, Trial=3, Global=5
----------  ----------  NEW ENVIRONMENT ----------  ----------
Dataset Environment    = 

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\specs\\Desktop\\Robotology\\Repos\\RORcogmaps/Models/Control-Exp3-NEpisodes100-NHidden200-BSize256-C1000.0-A100000.0-LR0.0001/Tmaze_v0a-Trial0of100-NHidden200-BSize256-C1000.0-A100000.0-LR0.0001.pth'